In [2]:
# ===== Cell 1: 检查本地 ChatGLM3 API 服务是否在线 =====
# 服务端启动方式（在终端中运行，不是在 notebook 里）：
#   cd S:\PythonAILearningCourse\15-transformers-basics\models\ChatGLM3-main\openai_api_demo
#   ..\..\..\.venv\Scripts\python.exe api_server.py
# api_server.py 已做本机适配：
#   ① subst X: 虚拟盘解决 SentencePiece 不支持中文路径
#   ② ChatGLM3-6B 以 4bit 量化加载（约3.8GB显存，8GB 显卡可完整装载）
#   ③ embedding 模型放 CPU，加载失败也不影响对话接口
# 服务启动后监听 http://localhost:8000，提供 OpenAI 兼容接口：
#   GET  /v1/models             查看模型
#   POST /v1/chat/completions   对话（本节课使用）
import requests

API_BASE = 'http://127.0.0.1:8000/v1'

health = requests.get('http://127.0.0.1:8000/health', timeout=5)
print('健康检查 /health ->', health.status_code)

models = requests.get(f'{API_BASE}/models', timeout=5).json()
print('可用模型:', [m['id'] for m in models['data']])



健康检查 /health -> 200
可用模型: ['chatglm3-6b']


In [3]:
# ===== Cell 2: 用 LangChain 接入本地模型（OpenAI 兼容协议）=====
# 关键点：本地 api_server.py 完全模仿 OpenAI 的接口格式，
# 所以 LangChain 里不用换任何代码，只要把 ChatOpenAI 的地址指向本机即可：
#   base_url  -> 本地服务地址（云端付费模型时是商汤地址）
#   api_key   -> 本地服务不鉴权，随便填一个占位即可
#   model     -> 服务端 /v1/models 列出的模型名 chatglm3-6b
import time
from langchain_openai import ChatOpenAI

local_llm = ChatOpenAI(
    model='chatglm3-6b',
    base_url=API_BASE,
    api_key='EMPTY',
    temperature=0.4,
    max_tokens=800,      # 对应服务端的 max_new_tokens（生成长度上限）
    timeout=300,
)

# 冒烟测试：确认 LangChain 能正常调用本地模型
t0 = time.time()
resp = local_llm.invoke([{'role': 'user', 'content': '用一句话介绍什么是导数。'}])
print(f'本地模型回答（{time.time()-t0:.1f}s）:')
print(resp.content)



本地模型回答（3.0s）:
导数是函数在某一点处的切线斜率,表示函数在该点的变化速率。


In [ ]:
# ===== Cell 3: 从 math 可视化页面解析 3 个知识点（扩充知识库的原料）=====
# 和第10课相同的解析方式：读取 HTML 的 <meta> 元数据
import os
from bs4 import BeautifulSoup

MATH_DIR = os.path.join(os.getcwd(), 'txt', 'math')
# 本地模型生成的文档单独存放，和云端模型生成的 44 篇区分开
OUT_DIR = os.path.join(os.getcwd(), 'txt', 'math_knowledge_local')
os.makedirs(OUT_DIR, exist_ok=True)


def parse_html_meta(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f.read(), 'html.parser')
    title = soup.title.string.strip() if soup.title else os.path.basename(filepath)
    name = title.split('·')[0].strip()
    meta = {t.get('name'): t.get('content', '').strip()
            for t in soup.find_all('meta') if t.get('name')}
    return {
        'name': name,
        'grade': meta.get('grade', ''),
        'topic': meta.get('topic', ''),
        'difficulty': meta.get('difficulty', ''),
        'priority': meta.get('priority', ''),
        'cognitive': meta.get('cognitive', ''),
        'desc': meta.get('desc', ''),
        'filename': os.path.basename(filepath),
    }


# 只挑 3 个知识点做演示（用户要求：只生成三个）
PICK_3 = ['黎曼和逼近积分.html', '马尔可夫链状态转移模拟器.html', '阻尼振动模拟器.html']
items = [parse_html_meta(os.path.join(MATH_DIR, f)) for f in PICK_3]

for it in items:
    print(f"- {it['name']}（{it['topic']} · 难度{it['difficulty']}）")
    print(f"    {it['desc'][:60]}")



In [ ]:
# ===== Cell 4: 调用本地 ChatGLM3 生成 3 篇知识点 Markdown 文档 =====
# 与第10课用付费云端模型不同，这次生成完全离线、不花 token 费用，
# 代价是速度慢一些（4bit 量化 + use_cache=False）。

DOC_PROMPT = '''你是数学老师。请根据以下信息，为学生编写一份结构清晰的知识点讲解文档(Markdown格式)。

知识点：{name}
学科分类：{grade} · {topic}
难度：{difficulty}
认知动作：{cognitive}
一句话描述：{desc}
配套工具：{filename}（一个可交互操作的可视化网页）

要求：
1. 直接输出 Markdown 正文，不要寒暄、不要用代码块包裹整篇内容，也不要重复输出一级标题（# 标题）。
2. 文档包含以下五个部分，总字数控制在 400~600 字：
   ## 一句话理解
   ## 核心概念（用通俗语言解释）
   ## 原理与公式（关键定义/定理/公式）
   ## 动手实验（如何操作配套可视化网页，至少3步）
   ## 学习要点（易错点或记忆口诀，2~4条）
3. 内容必须科学准确，围绕"{name}"展开。
'''


def build_markdown(item, body):
    # 模型偶尔会在正文开头再写一遍 "# 标题"，表头已有标题，自动删掉正文里的一级标题
    body = '\n'.join(ln for ln in body.splitlines() if not ln.lstrip().startswith('# ')).strip()
    header = (
        f'# {item["name"]}\n\n'
        f'> **{item["grade"]} · {item["topic"]}** ｜ '
        f'难度：{item["difficulty"]} ｜ 优先级：{item["priority"]} ｜ '
        f'认知动作：{item["cognitive"]}\n'
        f'> 配套可视化工具：`{item["filename"]}`\n'
        f'> 本文档由**本地 ChatGLM3-6B（4bit量化）**生成\n\n---\n\n'
    )
    return header + body


for i, item in enumerate(items, 1):
    print(f'[{i}/3] 本地模型正在生成「{item["name"]}」...')
    t0 = time.time()
    content = ''
    # 本地模型偶发输出过短，最多重试 3 次
    for attempt in range(3):
        resp = local_llm.invoke([{'role': 'user',
                                  'content': DOC_PROMPT.format(**item)}])
        content = resp.content.strip()
        if len(content) >= 150:
            break
        print(f'   第{attempt+1}次输出过短({len(content)}字)，重试...')
    md = build_markdown(item, content)
    out_path = os.path.join(OUT_DIR, item['filename'].replace('.html', '.md'))
    with open(out_path, 'w', encoding='utf-8') as f:
        f.write(md)
    print(f'   已保存 {os.path.basename(out_path)}（{len(md)}字, {time.time()-t0:.0f}s）')



In [ ]:
# ===== Cell 5: 验收本地模型扩充的知识库 =====
import glob

md_files = sorted(glob.glob(os.path.join(OUT_DIR, '*.md')))
print(f'本地模型生成的知识点文档: {len(md_files)} 个')
for p in md_files:
    size = os.path.getsize(p)
    print(f' - {os.path.basename(p)}（{size} 字节）')

# 抽查第一篇的开头
sample = md_files[0]
print(f'\n===== 抽查: {os.path.basename(sample)} =====')
with open(sample, 'r', encoding='utf-8') as f:
    print(f.read()[:600])

